# GEE ERA5-Land Downloader - Multi-Variable Hourly (Per-Variable Stacking)
Notebook ini didesain khusus untuk mengunduh data reanalisis atmosfer per-jam **ERA5-Land Hourly** (`ECMWF/ERA5_LAND/HOURLY`) dari Google Earth Engine (GEE) untuk 6 variabel atmosfer utama:
1. **`precipitation`**: Curah hujan (m -> **mm**, `x1000`)
2. **`temperature_2m`**: Suhu udara 2m (K -> **°C**, `-273.15`)
3. **`dewpoint_temperature_2m`**: Suhu titik embun 2m (K -> **°C**, `-273.15`)
4. **`u_wind_10m`**: Zonal wind / angin U 10m (**m/s**)
5. **`v_wind_10m`**: Meridional wind / angin V 10m (**m/s**)
6. **`surface_pressure`**: Tekanan permukaan (Pa -> **hPa**, `/100`)

### Fitur Utama:
- **Dual Compatibility:** Dapat berjalan di environment **Lokal** maupun **Kaggle Notebook** secara otomatis.
- **GeoJSON Boundary:** Memotong data sesuai poligon batas wilayah (`33.05_kecamatan.geojson`).
- **Smart Cache / Input Check:** Mendeteksi file yang sudah diunduh sebelumnya di Kaggle Input Datasets.
- **Per-Variable Stacking:** Mengunduh 6 variabel secara individual (744 band/var $\le$ 1024 limit GEE) lalu digabungkan menjadi 1 file NetCDF multi-variabel.
- **Struktur Hierarkis 4-Tier:** Menyiapkan folder `data/era5_land/<tahun>/` secara sistematis.

In [1]:
!pip install kagglehub geemap earthengine-api rioxarray geopandas xarray netCDF4 requests --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 568.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.7 MB/s eta 0:00:00


In [2]:
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import shutil
import glob
import rioxarray as rxr
import xarray as xr
import pandas as pd
import calendar
from datetime import datetime, timedelta

# ==========================================
# 0. PENGATURAN DIREKTORI DINAMIS & PARAMETER
# ==========================================
BASE_DIR = os.getcwd()

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    file_geojson = "/kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson"
    if not os.path.exists(file_geojson):
        found_geo = glob.glob("/kaggle/input/**/33.05_kecamatan.geojson", recursive=True)
        file_geojson = found_geo[0] if found_geo else os.path.join(BASE_DIR, "33.05_kecamatan.geojson")
else:
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

FOLDER_BASE_OUTPUT = os.path.join(BASE_DIR, "data", "era5_land")

# Direktori Dataset ERA5-Land yang sudah diunduh sebelumnya di Kaggle (Smart Cache)
KAGGLE_ERA5_INPUT = "/kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/data/era5_land"

tahun_awal = 2000
tahun_akhir = 2026

print(f"📌 File GeoJSON        : {file_geojson}")
print(f"📌 Folder Base Output  : {FOLDER_BASE_OUTPUT}")
print(f"📌 Folder Input Kaggle : {KAGGLE_ERA5_INPUT}")
print(f"📌 Periode Pengunduhan : {tahun_awal} s.d. {tahun_akhir}")


📌 File GeoJSON        : /kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson
📌 Folder Base Output  : /kaggle/working/data/era5_land
📌 Folder Input Kaggle : /kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/data/era5_land
📌 Periode Pengunduhan : 2000 s.d. 2026


In [3]:
# =========================================================================
# 1. INISIALISASI GOOGLE EARTH ENGINE (GEE)
# Mendukung: 1. Service Account Lokal (.json) | 2. Kaggle Secrets (GEE_KEY) | 3. Fallback Auth
# =========================================================================
import os
import json
import ee
from google.oauth2.service_account import Credentials

def init_earth_engine(project_id='staklimjerukagung'):
    # 1. Coba via Service Account File Lokal (Repo Projek_Downscale)
    sa_candidates = [
        os.path.join(os.getcwd(), "staklimjerukagung-b852a12a367e.json"),
        os.path.join(os.path.dirname(os.getcwd()), "staklimjerukagung-b852a12a367e.json"),
        "staklimjerukagung-b852a12a367e.json"
    ]
    for sa_path in sa_candidates:
        if os.path.exists(sa_path):
            try:
                with open(sa_path, 'r') as f:
                    sa_info = json.load(f)
                SCOPES = ['https://www.googleapis.com/auth/earthengine']
                credentials = Credentials.from_service_account_info(sa_info, scopes=SCOPES)
                ee.Initialize(credentials=credentials, project=project_id)
                print(f"✅ Berhasil Inisialisasi GEE via Local SA File: {os.path.basename(sa_path)}")
                return
            except Exception as e:
                print(f"⚠️ Gagal via {sa_path}: {e}")

    # 2. Coba via Kaggle Secrets (Jika di-run di Kaggle Kernel)
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        sa_info = json.loads(user_secrets.get_secret("GEE_KEY"))
        SCOPES = ['https://www.googleapis.com/auth/earthengine']
        credentials = Credentials.from_service_account_info(sa_info, scopes=SCOPES)
        ee.Initialize(credentials=credentials, project=project_id)
        print("✅ Berhasil Inisialisasi GEE via Kaggle Secret (GEE_KEY)")
        return
    except Exception:
        pass

    # 3. Fallback ke autentikasi default / browser interaktif
    try:
        ee.Initialize(project=project_id)
        print("✅ Berhasil Inisialisasi GEE via Default Session")
    except Exception as e:
        print(f"⚠️ Inisialisasi default gagal ({e}), membuka autentikasi browser...")
        ee.Authenticate()
        ee.Initialize(project=project_id)

init_earth_engine()


✅ Berhasil Inisialisasi GEE via Kaggle Secret (GEE_KEY)


In [4]:
# ==========================================
# 2. PERSIAPAN BATAS WILAYAH (GEOJSON CLEAN)
# ==========================================
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan di: {file_geojson}")

gdf = gpd.read_file(file_geojson)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

print("Membersihkan geometri GeoJSON yang cacat...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    if geom is None or geom.is_empty: return None
    return geom.buffer(0)

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

print("Mengonversi ke Earth Engine...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])
bounds = gdf.total_bounds
ee_bbox = ee.Geometry.BBox(bounds[0] - 0.01, bounds[1] - 0.01, bounds[2] + 0.01, bounds[3] + 0.01)
print(f"✅ Batas Wilayah Siap! Extent: Lon [{bounds[0]:.4f}, {bounds[2]:.4f}], Lat [{bounds[1]:.4f}, {bounds[3]:.4f}]")

Membersihkan geometri GeoJSON yang cacat...
Mengonversi ke Earth Engine...
✅ Batas Wilayah Siap! Extent: Lon [109.3788, 109.8393], Lat [-7.8312, -7.4457]


In [5]:
# ==========================================
# 3. FUNGSI UNDUH SUPER-CEPAT ERA5-LAND (PER-VARIABLE STACKING) DENGAN PRE-FLIGHT GUARDRAILS
# ==========================================
import calendar
import shutil
import xarray as xr

ERA5_EXPECTED_VARS = [
    'precipitation', 'temperature_2m', 'dewpoint_temperature_2m',
    'u_wind_10m', 'v_wind_10m', 'surface_pressure'
]

def check_file_integrity(nc_path, year, month, expected_type='era5_land'):
    """
    Tier 1: Validasi Integritas File NetCDF ERA5-Land.
    Memeriksa eksistensi, ukuran > 0, kecukupan 6 variabel atmosferik, dan total time steps (days * 24).
    """
    if not os.path.exists(nc_path):
        return False, "File belum ada", 0
    if os.path.getsize(nc_path) == 0:
        return False, "File 0 byte (kosong/korup)", 0

    _, days_in_month = calendar.monthrange(year, month)
    expected_steps = days_in_month * 24

    try:
        with xr.open_dataset(nc_path) as ds:
            if 'time' not in ds.dims and 'time' not in ds.coords:
                return False, "Dimensi 'time' tidak ditemukan", 0
            actual_steps = ds.sizes.get('time', len(ds['time']))
            missing_vars = [v for v in ERA5_EXPECTED_VARS if v not in ds.data_vars]
            if missing_vars:
                return False, f"Variabel hilang: {missing_vars}", actual_steps
            if actual_steps < expected_steps:
                return False, f"Time steps terpotong ({actual_steps}/{expected_steps} jam)", actual_steps
            return True, f"Lengkap ({actual_steps}/{expected_steps} jam, 6 vars)", actual_steps
    except Exception as e:
        return False, f"File rusak / korup ({e})", 0

def check_disk_space(target_dir, min_free_gb=1.5):
    """
    Tier 3: Disk Space Guardrail.
    Memeriksa sisa kapasitas penyimpanan sebelum ekspor citra.
    """
    try:
        _, _, free = shutil.disk_usage(target_dir)
        free_gb = free / (1024 ** 3)
        if free_gb < min_free_gb:
            print(f"  [DISK GUARD] Peringatan: Sisa ruang disk tinggal {free_gb:.2f} GB (< {min_free_gb} GB) di {target_dir}")
            return False, free_gb
        return True, free_gb
    except Exception:
        return True, 999.0

def cari_file_input_era5(tahun, bulan):
    """Mengecek apakah file NetCDF ERA5-Land (lengkap maupun parsial) tersedia di Input Dataset Kaggle."""
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        possible_paths = [
            f"{KAGGLE_ERA5_INPUT}/{tahun}/era5_land_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_ERA5_INPUT}/{tahun}/{tahun}_{bulan:02d}/era5_land_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/data/era5_land/{tahun}/era5_land_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/{tahun}/era5_land_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/era5_land_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/gee-era5-land-kebumen/data/era5_land/{tahun}/era5_land_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/gee-era5-land-kebumen/{tahun}/era5_land_{tahun}_{bulan:02d}.nc"
        ]
        best_path = None
        max_steps = 0
        for p in possible_paths:
            if os.path.exists(p):
                valid, msg, steps = check_file_integrity(p, tahun, bulan, 'era5_land')
                if valid:
                    return p, True, steps
                elif steps > max_steps:
                    max_steps = steps
                    best_path = p
        if best_path:
            return best_path, False, max_steps
    return None, False, 0

def unduh_era5_land_bulanan(tahun, bulan, batas_ee, output_dir, force=False):
    """
    Mengunduh 6 variabel atmosferik ERA5-Land per jam (hourly) dengan 4-tier Pre-Flight Guardrails & Atomic Writing.
    """
    os.makedirs(output_dir, exist_ok=True)
    nc_path = os.path.join(output_dir, f"era5_land_{tahun}_{bulan:02d}.nc")
    nc_path_tmp = f"{nc_path}.tmp"
    
    # 1. Tier 1: Cek di working directory
    is_valid, msg, actual_steps = check_file_integrity(nc_path, tahun, bulan, 'era5_land')
    if not force and is_valid:
        print(f"[{tahun}-{bulan:02d}] [OK] File sudah lengkap ({actual_steps} jam, 6 vars): {nc_path} (Dilewati)")
        return nc_path
        
    # 2. Cek di Kaggle Input Datasets (Smart Partial & Full Cache)
    if not force:
        input_file, is_cache_valid, cache_steps = cari_file_input_era5(tahun, bulan)
        if input_file and os.path.exists(input_file):
            if is_cache_valid:
                print(f"[{tahun}-{bulan:02d}] [KAGGLE CACHE 100%] File lengkap ditemukan di {input_file}. Menyalin...")
                shutil.copy2(input_file, nc_path)
                return nc_path
            elif cache_steps > actual_steps:
                print(f"[{tahun}-{bulan:02d}] [KAGGLE CACHE PARTIAL] Mengadopsi cache parsial ({cache_steps} jam) dari {input_file}...")
                shutil.copy2(input_file, nc_path)
                actual_steps = cache_steps

    if actual_steps > 0 and not is_valid:
        print(f"[{tahun}-{bulan:02d}] [PERIKSA GEE] File lokal terdeteksi tidak lengkap: {msg}. Memeriksa ketersediaan data di GEE...")
        
    tgl_mulai = f"{tahun}-{bulan:02d}-01"
    tgl_akhir = f"{tahun+1}-01-01" if bulan == 12 else f"{tahun}-{bulan+1:02d}-01"
        
    print(f"[{tahun}-{bulan:02d}] Menarik data ImageCollection ERA5-Land dari GEE...")
    temp_tifs = []
    try:
        col = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
               .filterBounds(ee_bbox)
               .filterDate(tgl_mulai, tgl_akhir))
        
        # Tier 2: GEE Pre-Flight Metadata Check
        count = col.size().getInfo()
        if count == 0:
            print(f"[{tahun}-{bulan:02d}] [INFO GEE] Data tidak tersedia di GEE untuk periode ini.")
            return None

        if not force and actual_steps > 0 and count <= actual_steps:
            print(f"[{tahun}-{bulan:02d}] [INFO GEE] File lokal sudah up-to-date dengan data GEE saat ini ({actual_steps}/{count} jam). Dilewati.")
            return nc_path

        # Tier 3: Disk Space Check
        check_disk_space(output_dir)
            
        timestamps = col.aggregate_array("system:time_start").getInfo()
        dates = pd.to_datetime(timestamps, unit='ms')
        
        # Konfigurasi 6 Variabel & Transformasi Satuan
        vars_config = [
            ('precipitation', 'total_precipitation_hourly', lambda img: img.multiply(1000)),
            ('temperature_2m', 'temperature_2m', lambda img: img.subtract(273.15)),
            ('dewpoint_temperature_2m', 'dewpoint_temperature_2m', lambda img: img.subtract(273.15)),
            ('u_wind_10m', 'u_component_of_wind_10m', lambda img: img),
            ('v_wind_10m', 'v_component_of_wind_10m', lambda img: img),
            ('surface_pressure', 'surface_pressure', lambda img: img.divide(100))
        ]
        
        ds_out = xr.Dataset()
        
        for var_out_name, gee_band_name, transform_fn in vars_config:
            tif_var_temp = os.path.join(output_dir, f"temp_era5_{var_out_name}_{tahun}_{bulan:02d}.tif")
            temp_tifs.append(tif_var_temp)
            
            col_var = col.select(gee_band_name).map(lambda img: transform_fn(img).rename(var_out_name))
            stacked_var_img = col_var.toBands().clip(ee_bbox)
            
            # Export per variabel (max 744 band <= 1024)
            print(f"[{tahun}-{bulan:02d}] Mengekspor {var_out_name} ({count} jam <= 1024) ke TIF temporer...")
            geemap.ee_export_image(
                stacked_var_img,
                filename=tif_var_temp,
                region=ee_bbox,
                scale=11132,
                file_per_band=False
            )
            
            with rxr.open_rasterio(tif_var_temp, masked=True) as da_var:
                da_var = da_var.rename({'band': 'time'})
                da_var['time'] = dates[:len(da_var.time)]
                da_var.name = var_out_name
                ds_out[var_out_name] = da_var.load()
                
        # Tier 4: Atomic NetCDF Staging & Verification
        ds_out.attrs["source"] = "ECMWF/ERA5_LAND/HOURLY"
        ds_out.attrs["temporal_resolution"] = "Hourly"
        ds_out.to_netcdf(nc_path_tmp)
        ds_out.close()
        
        for temp_tif in temp_tifs:
            if os.path.exists(temp_tif):
                os.remove(temp_tif)

        tmp_valid, tmp_msg, tmp_steps = check_file_integrity(nc_path_tmp, tahun, bulan, 'era5_land')
        if tmp_valid or (tmp_steps == count):
            if os.path.exists(nc_path):
                os.remove(nc_path)
            os.replace(nc_path_tmp, nc_path)
            print(f"[{tahun}-{bulan:02d}] [SUKSES] Tersimpan secara atomic: {nc_path} ({tmp_steps} jam, 6 vars)\n")
            return nc_path
        else:
            print(f"[{tahun}-{bulan:02d}] [ERROR] Validasi atomic gagal ({tmp_msg}). File sementara dibuang.")
            if os.path.exists(nc_path_tmp):
                os.remove(nc_path_tmp)
            return None

    except Exception as e:
        print(f"[{tahun}-{bulan:02d}] [ERROR]: {e}")
        for temp_tif in temp_tifs:
            if os.path.exists(temp_tif):
                os.remove(temp_tif)
        if os.path.exists(nc_path_tmp):
            os.remove(nc_path_tmp)
        return None

def unduh_era5_multi_tahun(tahun_awal, tahun_akhir, batas_ee, output_base_dir, force=False):
    print(f"\n{'='*60}")
    print(f"UNDUH ERA5-LAND HOURLY MULTI-VARIABEL: {tahun_awal} - {tahun_akhir} (Force: {force})")
    print(f"{'='*60}\n")
    
    for tahun in range(tahun_awal, tahun_akhir + 1):
        folder_tahun = os.path.join(output_base_dir, str(tahun))
        for bulan in range(1, 13):
            unduh_era5_land_bulanan(tahun, bulan, batas_ee, folder_tahun, force=force)
            
    print(f"{'='*60}")
    print("SELESAI UNDUH SELURUH TAHUN ERA5-LAND")
    print(f"{'='*60}")


In [6]:
# ==========================================
# 4. EKSEKUSI FUNGSI UTAMA ERA5-LAND
# ==========================================
unduh_era5_multi_tahun(
    tahun_awal=tahun_awal,
    tahun_akhir=tahun_akhir,
    batas_ee=batas_kebumen,
    output_base_dir=FOLDER_BASE_OUTPUT
)



UNDUH ERA5-LAND HOURLY MULTI-VARIABEL: 2000 - 2026 (Force: False)

[2000-01] Menarik data ImageCollection ERA5-Land dari GEE...
[2000-01] Mengekspor precipitation (744 jam <= 1024) ke TIF temporer...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/era5_land/2000/temp_era5_precipitation_2000_01.tif


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[2000-01] Mengekspor temperature_2m (744 jam <= 1024) ke TIF temporer...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/era5_land/2000/temp_era5_temperature_2m_2000_01.tif


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[2000-01] Mengekspor dewpoint_temperature_2m (744 jam <= 1024) ke TIF temporer...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/era5_land/2000/temp_era5_dewpoint_temperature_2m_2000_01.tif


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


... (219 outputs left out)

Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


Please wait ...
Data downloaded to /kaggle/working/data/era5_land/2026/temp_era5_surface_pressure_2026_11.tif
[2026-11] [SUKSES] Tersimpan secara atomic: /kaggle/working/data/era5_land/2026/era5_land_2026_11.nc (54 jam, 6 vars)



Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[2026-12] Menarik data ImageCollection ERA5-Land dari GEE...
[2026-12] [INFO GEE] Data tidak tersedia di GEE untuk periode ini.
SELESAI UNDUH SELURUH TAHUN ERA5-LAND


In [7]:
# =========================================================================
# UPLOAD OTOMATIS KE KAGGLE DATASET (KAGGLEHUB)
# =========================================================================
import os
import glob
import json
from datetime import datetime
import kagglehub
from kagglehub import auth

def init_kaggle_auth():
    """Menginisialisasi kredensial Kaggle untuk upload versi baru dataset."""
    # 1. Cek dari Kaggle Secrets (jika running di Kaggle Kernel)
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        try:
            from kaggle_secrets import UserSecretsClient
            secrets = UserSecretsClient()
            k_user = None
            k_key = None
            for u_label in ["KAGGLE_USERNAME", "kaggle_username", "USERNAME"]:
                try:
                    k_user = secrets.get_secret(u_label)
                    if k_user: break
                except Exception:
                    pass
            for k_label in ["KAGGLE_KEY", "kaggle_key", "KAGGLE_API_KEY", "KAGGLE_TOKEN"]:
                try:
                    k_key = secrets.get_secret(k_label)
                    if k_key: break
                except Exception:
                    pass
            
            if not k_user:
                k_user = "jerismeteo"
            
            if k_key:
                os.environ["KAGGLE_USERNAME"] = k_user
                os.environ["KAGGLE_KEY"] = k_key
                auth.set_kaggle_credentials(k_user, k_key)
                print(f"[AUTH] Kredensial Kaggle aktif via Secrets untuk user: {k_user}")
                return True
        except Exception as e:
            print(f"[AUTH] Kaggle Secrets tidak tersedia: {e}")

    # 2. Cek file kaggle.json lokal
    candidates = [
        os.path.join(os.getcwd(), "kaggle.json"),
        os.path.join(os.path.dirname(os.getcwd()), "kaggle.json"),
        os.path.expanduser("~/.kaggle/kaggle.json")
    ]
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    c = json.load(f)
                os.environ["KAGGLE_USERNAME"] = c["username"]
                os.environ["KAGGLE_KEY"] = c["key"]
                auth.set_kaggle_credentials(c["username"], c["key"])
                print(f"[AUTH] Kredensial Kaggle aktif via file: {p}")
                return True
            except Exception:
                pass

    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True

    print("[AUTH] Peringatan: Kredensial Kaggle (KAGGLE_KEY) belum terkonfigurasi.")
    return False

def upload_ke_kaggle(dataset_handle, upload_root, subfolder_data, nama_dataset):
    """
    Mengunggah dataset NetCDF terbaru ke Kaggle Dataset menggunakan kagglehub.
    Menghitung jumlah file .nc valid dan mengabaikan file temporer/cache.
    """
    if not init_kaggle_auth():
        print(f"⚠️ [UPLOAD] Dilewati: Kredensial Kaggle tidak ditemukan. Tambahkan Secret 'KAGGLE_KEY' di Kaggle Secrets.")
        return False
        
    target_nc_dir = os.path.join(upload_root, subfolder_data)
    list_files = glob.glob(os.path.join(target_nc_dir, "**", "*.nc"), recursive=True)
    if not list_files:
        print(f"⚠️ [UPLOAD] Tidak ditemukan file NetCDF di: {target_nc_dir}. Upload dibatalkan.")
        return False

    now_str = datetime.now().strftime("%Y-%m-%d %H:%M WIB")
    version_notes = f"Pembaruan Otomatis Data GEE ({nama_dataset}): {len(list_files)} file NetCDF per {now_str}"

    print(f"\n{'='*75}")
    print(f"🚀 MENGUNGGAH VERSI BARU KE KAGGLE DATASET")
    print(f"🏷️  Handle Dataset : {dataset_handle}")
    print(f"📁 Folder Basis   : {upload_root}")
    print(f"📊 Total File     : {len(list_files)} file NetCDF ({subfolder_data})")
    print(f"📝 Catatan Versi  : {version_notes}")
    print(f"{'='*75}")

    try:
        kagglehub.dataset_upload(
            handle=dataset_handle,
            local_dataset_dir=upload_root,
            version_notes=version_notes,
            ignore_patterns=[
                "*.tmp", "*.tif", "*.temp", "*.geojson", "*.json",
                "*.ipynb", "__pycache__/", ".ipynb_checkpoints/"
            ]
        )
        print(f"✅ [SUKSES] Dataset {dataset_handle} berhasil diperbarui di Kaggle!")
        return True
    except Exception as e:
        print(f"❌ [GAGAL] Terjadi kesalahan saat mengunggah dataset: {e}")
        return False

# Eksekusi upload otomatis (Hanya aktif jika berjalan di Kaggle Kernel atau FORCE_UPLOAD=True)
FORCE_UPLOAD = False

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or FORCE_UPLOAD:
    upload_ke_kaggle(
        dataset_handle="jerismeteo/gee-era5-land-kebumen",
        upload_root=BASE_DIR,
        subfolder_data='data/era5_land',
        nama_dataset="ECMWF ERA5-Land Kebumen"
    )
else:
    print("ℹ️ Eksekusi di environment lokal: Upload otomatis ke Kaggle Dataset dilewati.")
    print("💡 Untuk menguji upload lokal, ubah FORCE_UPLOAD = True.")

[AUTH] Peringatan: Kredensial Kaggle (KAGGLE_KEY) belum terkonfigurasi.
⚠️ [UPLOAD] Dilewati: Kredensial Kaggle tidak ditemukan. Tambahkan Secret 'KAGGLE_KEY' di Kaggle Secrets.
